#Databricks R Lab
<br>

Databricks Notebooks support more than Python and SQL. 
<br>By supporting multiple langagues Databricks allows users to user their preffered langague while at the same time preventing data copies, enabling data sharing and reducing costs.

In this lab R will be use to access tables in Unity Catalog, perform an analysis and save the result as a new table in Unity Catalog.



In [0]:
%run "../Lab 0 - Setup/SetCatalogName"

##Connecting R to Spark

Sparklyr is an R interface for Apache Spark. It allows R users to analyze and manipulate large datasets using dplyr, a popular data manipulation package in R. Sparklyr provides an intuitive interface to Spark's distributed computing capabilities, enabling users to perform data analysis at scale. With Sparklyr, you can connect to Spark clusters, access Spark DataFrames, and use Spark's machine learning library, all within the R environment.

First the sparklyr and dplyr packages are loaded.

In [0]:
library(sparklyr)
library(dplyr)

### Using `spark_connect` in R in Databricks

To connect to a Spark cluster from R in Databricks, you can use the `spark_connect` function from the `sparklyr` package. This function establishes a connection to a Spark cluster, allowing you to interact with Spark DataFrames and perform distributed data analysis.

To authenicate to Unity Catalog the `method = "databricks"` is used. The user's identity is used when requesting access to data in Unity Catalog. This eliminates the need for the user to configure any credentials that R users often need to do when connecting to data sources.

In [0]:
# Establish a connection to Spark
sc <- spark_connect(method = "databricks")

# Verify the connection
spark_connection_is_open(sc)

With a connection created read the tsms_claims tables from Unity Catalog. The reading of the table is done by Spark. Instead of a table if a query is used then the performance of distributed computing provided by Spark is used. The final result set is returned as a R dataframe.

In [0]:
tmsis_claims <- tbl(sc, "silver.tmsis_claims")

head(tmsis_claims)


Next several different statistics are computed for first diagnosis code. This is done using standard R dataframes so all processing is done a single node.

In [0]:

result <- tmsis_claims %>%
  group_by(DGNS_CD_1) %>%
  summarise(
    min_claim_rec_dt = min(TOT_MDCD_PD_AMT),
    max_claim_rec_dt = max(TOT_MDCD_PD_AMT),
    avg_claim_rec_dt = mean(TOT_MDCD_PD_AMT),
    sd_claim_rec_dt = sd(TOT_MDCD_PD_AMT)
  )

result


Now the R dataframe containing the statistics, result is saved back to Unity Catalog. <br>
The `spark_write_table` function takes a R dataframe, the name of the table and an optional parameter of what to do if the table already exists, in this case overwrite it.

In [0]:
spark_write_table(result, "silver.tmsis_stats", mode = "overwrite")


What about situations where you need to join large tables together or other complex SQL? Using R dataframes would be very slow if the operation even completed and didn't fail due to out of memory errors.

<br>Using Databricks ability to use multiple langagues in a single notebook provides an elegant way to have Spark due the heavy data processing. In the next cell `%sql` is used to switch the cell to SQL than a temporary view which is only available to this notebook is created using ANSI SQL.

In [0]:
%sql

create or replace temp view dgns_by_gndr as
select dgns_cd_1, gndr_cd, avg(TOT_MDCD_PD_AMT) avg_TOT_MDCD_PD_AMT, min(TOT_MDCD_PD_AMT) min_TOT_MDCD_PD_AMT, max(TOT_MDCD_PD_AMT) max_TOT_MDCD_PD_AMT, stddev(TOT_MDCD_PD_AMT) sd_TOT_MDCD_PD_AMT
from silver.tmsis_claims
group by all

With the temporary view created we can reference the view as if it was a table. The view will be executed by the Spark cluster and the results will be returned as a R dataframe.

In [0]:
claims_dg_gnd <- tbl(sc, "dgns_by_gndr")

head(claims_dg_gnd)

####Congratulations on Completing the Databricks R Lab